In [1]:
# Copyright (c) CIIS-Lab. All rights reserved.
import os.path as osp
import os
from pathlib import Path
import gc
import copy as cp
import tempfile

import cv2
import mmcv
import mmengine
import numpy as np
import torch

from mmaction.apis import (detection_inference,
                           # inference_recognizer, init_recognizer,
                           pose_inference)
from mmaction.registry import VISUALIZERS
from mmaction.utils import frame_extract

import moviepy.editor as mpy
import glob
import os.path as osp


In [2]:
FONTFACE = cv2.FONT_HERSHEY_DUPLEX
FONTSCALE = 1

THICKNESS = 1  # int
LINETYPE = 1

In [3]:
# def load_label_map(file_path):
#     """Load Label Map.

#     Args:
#         file_path (str): The file path of label map.

#     Returns:
#         dict: The label map (int -> label name).
#     """
#     lines = open(file_path).readlines()
#     lines = [x.strip().split(': ') for x in lines]
#     return {int(x[0]): x[1] for x in lines}


def abbrev(name):
    """Get the abbreviation of label name:

    'take (an object) from (a person)' -> 'take ... from ...'
    """
    while name.find('(') != -1:
        st, ed = name.find('('), name.find(')')
        name = name[:st] + '...' + name[ed + 1:]
    return name

def pack_result(human_detection, result, img_h, img_w):
    """Short summary.

    Args:
        human_detection (np.ndarray): Human detection result.
        result (type): The predicted label of each human proposal.
        img_h (int): The image height.
        img_w (int): The image width.

    Returns:
        tuple: Tuple of human proposal, label name and label score.
    """
    human_detection[:, 0::2] /= img_w
    human_detection[:, 1::2] /= img_h
    results = []
    if result is None:
        return None
    for prop, res in zip(human_detection, result):
        res.sort(key=lambda x: -x[1])
        results.append(
            (prop.data.cpu().numpy(), [x[0] for x in res], [x[1]
                                                            for x in res]))
    return results


def expand_bbox(bbox, h, w, ratio=1.25):
    x1, y1, x2, y2 = bbox
    center_x = (x1 + x2) // 2
    center_y = (y1 + y2) // 2
    width = x2 - x1
    height = y2 - y1

    square_l = max(width, height)
    new_width = new_height = square_l * ratio

    new_x1 = max(0, int(center_x - new_width / 2))
    new_x2 = min(int(center_x + new_width / 2), w)
    new_y1 = max(0, int(center_y - new_height / 2))
    new_y2 = min(int(center_y + new_height / 2), h)
    return (new_x1, new_y1, new_x2, new_y2)


def cal_iou(box1, box2):
    xmin1, ymin1, xmax1, ymax1 = box1
    xmin2, ymin2, xmax2, ymax2 = box2

    s1 = (xmax1 - xmin1) * (ymax1 - ymin1)
    s2 = (xmax2 - xmin2) * (ymax2 - ymin2)

    xmin = max(xmin1, xmin2)
    ymin = max(ymin1, ymin2)
    xmax = min(xmax1, xmax2)
    ymax = min(ymax1, ymax2)

    w = max(0, xmax - xmin)
    h = max(0, ymax - ymin)
    intersect = w * h
    union = s1 + s2 - intersect
    iou = intersect / union

    return iou


# clip_pose_extraction
def skeleton_based_stdet(predict_stepsize, video,
                         # skeleton_config, skeleton_stdet_checkpoint, device, action_score_thr, label_map,
                         human_detections, pose_results, num_frame, clip_len, frame_interval, h, w):
    window_size = clip_len * frame_interval
    assert clip_len % 2 == 0, 'We would like to have an even clip_len'
    timestamps = np.arange(window_size // 2, num_frame + 1 - window_size // 2,
                           predict_stepsize)

    # skeleton_config = mmengine.Config.fromfile(skeleton_config)
    # num_class = max(label_map.keys()) + 1  # for AVA dataset (81)
    # skeleton_config.model.cls_head.num_classes = num_class
    # skeleton_stdet_model = init_recognizer(skeleton_config,
    #                                        skeleton_stdet_checkpoint,
    #                                        device)

    skeleton_predictions = []
    skeleton_datasets = []

    print('Building skeleton datasets from existing keypoint data for each clip')
    prog_bar = mmengine.ProgressBar(len(timestamps))
    for timestamp in timestamps:  # iterate each clip
        proposal = human_detections[timestamp - 1] # get bboxes for persons in timestamp (first frame of clip)
        if proposal.shape[0] == 0:  # no people detected
            skeleton_predictions.append(None)
            continue

        start_frame = timestamp - (clip_len // 2 - 1) * frame_interval
        frame_inds = start_frame + np.arange(0, window_size, frame_interval)
        frame_inds = list(frame_inds - 1)
        num_frame = len(frame_inds)  # 30

        pose_result = [pose_results[ind] for ind in frame_inds]  # grouping frames poses for each clip

        skeleton_prediction = []
        for i in range(proposal.shape[0]):  # num_person  # iterate each bbox in timestamp (first frame of clip)
            skeleton_prediction.append([])

            fake_anno = dict(
                frame_dir=osp.splitext(osp.basename(video))[0]+"_"+str(timestamp+(i+1)*0.001),
                label=-1,
                img_shape=(h, w),
                original_shape=(h, w),
                num_clips=1,
                total_frames=num_frame
            )
            num_person = 1

            num_keypoint = 17
            keypoint = np.zeros(
                (num_person, num_frame, num_keypoint, 2))  # M T V 2
            keypoint_score = np.zeros(
                (num_person, num_frame, num_keypoint))  # M T V

            # pose matching
            person_bbox = proposal[i][:4]  # get bbox for a person in timestamp (first frame of clip)
            area = expand_bbox(person_bbox, h, w)  # bbox expanded by 1.25 ratio with square shape

            for j, poses in enumerate(pose_result):  # num_frame  # iterate each frame of clip
                max_iou = float('-inf')
                index = -1
                if len(poses['keypoints']) == 0:
                    continue
                for k, bbox in enumerate(poses['bboxes']):  # iterate each bbox/pose in each frame
                    iou = cal_iou(bbox, area)  # compare each bbox in each frame with current area (calculate_intersect/union)
                    if max_iou < iou:
                        index = k  # pose from the biggest intersect/union (iou) will be considered
                        max_iou = iou
                keypoint[0, j] = poses['keypoints'][index]
                keypoint_score[0, j] = poses['keypoint_scores'][index]

            fake_anno['keypoint'] = keypoint
            fake_anno['keypoint_score'] = keypoint_score

            skeleton_datasets.append(fake_anno)
            # output = inference_recognizer(skeleton_stdet_model, fake_anno)
            # # for multi-label recognition
            # score = output.pred_score.tolist()
            # for k in range(len(score)):  # 81
            #     if k not in label_map:
            #         continue
            #     if score[k] > action_score_thr:
            #         skeleton_prediction[i].append((label_map[k], score[k]))
            skeleton_prediction[i].append(("annotate!", timestamp + (i+1)*0.001))

        skeleton_predictions.append(skeleton_prediction)
        prog_bar.update()

    return timestamps, skeleton_predictions, skeleton_datasets

In [4]:
def hex2color(h):
    """Convert the 6-digit hex string to tuple of 3 int value (RGB)"""
    return (int(h[:2], 16), int(h[2:4], 16), int(h[4:], 16))

PLATEBLUE = '03045e-023e8a-0077b6-0096c7-00b4d8-48cae4'
PLATEBLUE = PLATEBLUE.split('-')
PLATEBLUE = [hex2color(h) for h in PLATEBLUE]


def visualize(pose_config,
              frames,
              annotations,
              pose_data_samples,
              action_result,
              plate=PLATEBLUE,
              max_num=5):
    """Visualize frames with predicted annotations.

    Args:
        frames (list[np.ndarray]): Frames for visualization, note that
            len(frames) % len(annotations) should be 0.
        annotations (list[list[tuple]]): The predicted spatio-temporal
            detection results.
        pose_data_samples (list[list[PoseDataSample]): The pose results.
        action_result (str): The predicted action recognition results.
        pose_model (nn.Module): The constructed pose model.
        plate (str): The plate used for visualization. Default: PLATEBLUE.
        max_num (int): Max number of labels to visualize for a person box.
            Default: 5.

    Returns:
        list[np.ndarray]: Visualized frames.
    """

    assert max_num + 1 <= len(plate)
    frames_ = cp.deepcopy(frames)
    frames_ = [mmcv.imconvert(f, 'bgr', 'rgb') for f in frames_]
    nf, na = len(frames), len(annotations)
    assert nf % na == 0
    nfpa = len(frames) // len(annotations)
    anno = None
    h, w, _ = frames[0].shape
    scale_ratio = np.array([w, h, w, h])

    # add pose results
    if pose_data_samples:
        pose_config = mmengine.Config.fromfile(pose_config)
        visualizer = VISUALIZERS.build(pose_config.visualizer | {'line_width':5, 'bbox_color':(101,193,255), 'radius': 8})  # https://mmpose.readthedocs.io/en/latest/api.html#mmpose.visualization.PoseLocalVisualizer
        visualizer.set_dataset_meta(pose_data_samples[0].dataset_meta)
        for i, (d, f) in enumerate(zip(pose_data_samples, frames_)):
            visualizer.add_datasample(
                'result',
                f,
                data_sample=d,
                draw_gt=False,
                draw_heatmap=False,
                draw_bbox=True,
                draw_pred=True,
                show=False,
                wait_time=0,
                out_file=None,
                kpt_thr=0.3)
            frames_[i] = visualizer.get_image()

    for i in range(na):
        anno = annotations[i]
        if anno is None:
            continue
        for j in range(nfpa):
            ind = i * nfpa + j
            frame = frames_[ind]

            # add spatio-temporal action detection results
            for ann in anno:
                box = ann[0]
                label = ann[1]
                if not len(label):
                    continue
                score = ann[2]
                box = (box * scale_ratio).astype(np.int64)
                st, ed = tuple(box[:2]), tuple(box[2:])
                if not pose_data_samples:
                    cv2.rectangle(frame, st, ed, plate[0], 2)

                for k, lb in enumerate(label):
                    if k >= max_num:
                        break
                    text = abbrev(lb)
                    text = ': '.join([text, f'{score[k]:.3f}'])
                    location = (0 + st[0], 18 + k * 18 + st[1])
                    textsize = cv2.getTextSize(text, FONTFACE, FONTSCALE,
                                               THICKNESS)[0]
                    textwidth = textsize[0]
                    diag0 = (location[0] + textwidth, location[1] - 14)
                    diag1 = (location[0], location[1] + 2)
                    cv2.rectangle(frame, diag0, diag1, plate[k + 1], -1)
                    FONTCOLOR = (255, 0, 0)
                    cv2.putText(frame, text, location, FONTFACE, FONTSCALE,
                                FONTCOLOR, THICKNESS, LINETYPE)

    return frames_

In [5]:
# video_folder_path = osp.abspath("/media/ciis/680d5156-2214-4b41-baf0-6bb993bff707/ciis-compnew/Documents/ActionTracking/video/")
extracted_folder_path = "../dataset/2025/sampleFrame_4_out_16fps"
video_folder_path = "../assets/video/25_5_v1"

excluded_file = ['lab_7']
final_files = sorted([
        f for f in os.listdir(extracted_folder_path)
        # Check if it IS a directory
        if os.path.isdir(os.path.join(extracted_folder_path, f)) 
        # AND check if its base name (without extension) is NOT excluded
        and os.path.splitext(f)[0] not in excluded_file
    ])

    # 'final_files' now holds only the directory names that meet the criteria


print(str(final_files))


['lab_1_1', 'lab_1_2', 'lab_1_3', 'lab_2_1', 'lab_2_2', 'lab_2_3', 'lab_3_1', 'lab_3_2', 'lab_3_3', 'lab_4_1', 'lab_4_2', 'lab_4_3', 'lab_5_1', 'lab_5_2', 'lab_6_1', 'lab_6_2', 'lab_6_3', 'lab_7_1', 'lab_7_2', 'lab_7_3', 'lab_7_4', 'lab_7_5']


In [21]:
for folder in final_files:
    vidio_path = f'/{folder}'
    target_dir = str(extracted_folder_path) + str(vidio_path)
    video = str(video_folder_path) + str(vidio_path) + '.mp4'
    num_ver = 3

    path_extractPose = f'../dataset/2025/2025_{num_ver}/extracted_pose'
    path_annData = f'../dataset/2025/2025_{num_ver}/annotated_data'
    Path(path_extractPose).mkdir(parents=True, exist_ok=True)
    Path(path_annData).mkdir(parents=True, exist_ok=True)
        

    ann_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.csv'
    pkl_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.pkl'
    out_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '_gt.mp4'

    print(target_dir)
    print(video)
    print(pkl_filename)
    print(f'{extracted_folder_path}{vidio_path}.pkl')

../dataset/2025/sampleFrame_4_out_16fps/lab_1
../assets/video/25_5_v1/lab_1.mp4
../dataset/2025/2025_3/extracted_pose/lab_1.pkl
../dataset/2025/sampleFrame_4_out_16fps/lab_1.pkl
../dataset/2025/sampleFrame_4_out_16fps/lab_2
../assets/video/25_5_v1/lab_2.mp4
../dataset/2025/2025_3/extracted_pose/lab_2.pkl
../dataset/2025/sampleFrame_4_out_16fps/lab_2.pkl
../dataset/2025/sampleFrame_4_out_16fps/lab_3
../assets/video/25_5_v1/lab_3.mp4
../dataset/2025/2025_3/extracted_pose/lab_3.pkl
../dataset/2025/sampleFrame_4_out_16fps/lab_3.pkl
../dataset/2025/sampleFrame_4_out_16fps/lab_4
../assets/video/25_5_v1/lab_4.mp4
../dataset/2025/2025_3/extracted_pose/lab_4.pkl
../dataset/2025/sampleFrame_4_out_16fps/lab_4.pkl
../dataset/2025/sampleFrame_4_out_16fps/lab_5
../assets/video/25_5_v1/lab_5.mp4
../dataset/2025/2025_3/extracted_pose/lab_5.pkl
../dataset/2025/sampleFrame_4_out_16fps/lab_5.pkl
../dataset/2025/sampleFrame_4_out_16fps/lab_6
../assets/video/25_5_v1/lab_6.mp4
../dataset/2025/2025_3/extract

In [10]:
# human detection config
det_config = "../mmaction2/demo/demo_configs/faster-rcnn_r50_fpn_2x_coco_infer.py"
det_checkpoint = 'http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth'
det_score_thr = 0.9
#det_cat_id = 0

# pose estimation config
pose_config = '../mmaction2/demo/demo_configs/td-hm_hrnet-w32_8xb64-210e_coco-256x192_infer.py'
pose_checkpoint = 'https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth'

# use skeleton-based method
use_skeleton_stdet = True
use_skeleton_recog = True

# skeleton-based spatio-temporal action classification config
label_map_stdet = "../mmaction2/tools/data/ciis/ciis_label_map.txt"

predict_stepsize = 4  # must even int, give out a spatio-temporal detection prediction per n frames
output_stepsize = 1  # show one frame per n frames in the demo, we should have: predict_stepsize % output_stepsize == 0, speedUp/slowDown video output
output_fps = 12  # the fps of demo video output, will speedUp/slowDown video output, must equal to (video_input_fps/output_stepsize) to get normal speed

device = 'cuda'

### Lo cek diri 

In [23]:
for folder in final_files:
    vidio_path = f'/{folder}'
    target_dir = str(extracted_folder_path) + str(vidio_path)
    video = str(video_folder_path) + str(vidio_path) + '.mp4'
    num_ver = 3

    path_extractPose = f'../dataset/2025/2025_{num_ver}/extracted_pose'
    path_annData = f'../dataset/2025/2025_{num_ver}/annotated_data'
    Path(path_extractPose).mkdir(parents=True, exist_ok=True)
    Path(path_annData).mkdir(parents=True, exist_ok=True)
        

    ann_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.csv'
    pkl_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.pkl'
    out_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '_gt.mp4'

    if not osp.exists(target_dir):
        print(f"Error: The specified directory does not exist: {target_dir}")
    else:
        # Create the search pattern to find all 'img_*.jpg' files
        # directly within the target_dir.
        search_pattern = osp.join(target_dir, 'img_*.jpg')

        # Use glob.glob to find all matching file paths
        frame_paths = glob.glob(search_pattern)

        # Sort the paths to ensure they are in frame order
        frame_paths.sort()

        # Print the results
        if frame_paths:
            print(f"Found {len(frame_paths)} frames in '{target_dir}':")
            # Print the first few and the last one as an example
            for path in frame_paths[:5]: # Print first 5
                print(path)
            if len(frame_paths) > 5:
                print("...")
                print(frame_paths[-1]) # Print the last one
        else:
            print(f"No frames matching 'img_*.jpg' were found in '{target_dir}'.")

    num_frame=len(frame_paths)
    original_frames = cv2.imread(frame_paths[0])
    #print(original_frames.shape)
    h, w, _ = original_frames.shape


    # get Human detection results
    human_detections, _ = detection_inference(
        det_config,
        det_checkpoint,
        frame_paths,
        det_score_thr,
        device=device)
    torch.cuda.empty_cache()

    # get Pose estimation results
    pose_datasample = None
    pose_results, pose_datasample = pose_inference(
        pose_config,
        pose_checkpoint,
        frame_paths,
        human_detections,
        device=device)
    torch.cuda.empty_cache()

    stdet_preds = None

    print('Use skeleton-based SpatioTemporal Action Detection')
    # clip_len, frame_interval = 30, 1
    clip_len, frame_interval = predict_stepsize, 1

    # clip_pose_extraction
    timestamps, stdet_preds, skeleton_datasets = skeleton_based_stdet(predict_stepsize, video,
                                                                    # skeleton_config,
                                                                    # skeleton_stdet_checkpoint,
                                                                    # device,
                                                                    # action_score_thr,
                                                                    # stdet_label_map,
                                                                    human_detections,
                                                                    pose_results, num_frame,
                                                                    clip_len,
                                                                    frame_interval, h, w)
    for i in range(len(human_detections)):
        det = human_detections[i]
        # det[:, 0:4:2] *= w_ratio
        # det[:, 1:4:2] *= h_ratio
        det[:, 0:4:2] *= 1
        det[:, 1:4:2] *= 1
        human_detections[i] = torch.from_numpy(det[:, :4]).to(device)

    anno = ""
    for clip in stdet_preds:
        if clip == None:
            continue
        for person_attr in clip:
            anno += str(person_attr[0][0]) + "," + str(person_attr[0][1]) + "\n"

    with open(ann_filename,'w') as data:
        data.write(anno)

    mmengine.dump(skeleton_datasets, pkl_filename)

    stdet_results = []
    for timestamp, prediction in zip(timestamps, stdet_preds):
        human_detection = human_detections[timestamp - 1]
        stdet_results.append(
            pack_result(human_detection, prediction, h, w))

    def dense_timestamps(timestamps, n):
        """Make it nx frames."""
        old_frame_interval = (timestamps[1] - timestamps[0])
        start = timestamps[0] - old_frame_interval / n * (n - 1) / 2
        new_frame_inds = np.arange(
            len(timestamps) * n) * old_frame_interval / n + start
        return new_frame_inds.astype(np.int64)

    dense_n = int(predict_stepsize / output_stepsize)
    output_timestamps = dense_timestamps(timestamps, dense_n) + 1
    # frames = [
    #     cv2.imread(frame_paths[timestamp - 1])
    #     for timestamp in output_timestamps
    # ]

    pose_datasample = [
        pose_datasample[timestamp - 1] for timestamp in output_timestamps
    ]
    mmengine.dump(
    {
        'stdet_results': stdet_results,
        'pose_datasample': pose_datasample,
        'output_timestamps': output_timestamps
    },
    f'{extracted_folder_path}{vidio_path}.pkl'
)

    print(f"Finished processing {folder}.")

    # --------------------------------------------------
    # --- 🧹 START: Memory Clearing Section ---
    # --------------------------------------------------
    print("Clearing memory for next iteration...")

    # 1. Explicitly delete large variables from this iteration
    try:
        del frame_paths
        del original_frames
        del human_detections
        del pose_results
        del pose_datasample
        del stdet_preds
        del skeleton_datasets
        # del stdet_results # Uncomment if you define this
        # del anno # Uncomment if you define this
    except NameError:
        print("Some variables were not defined, skipping deletion.")

    # 2. Suggest Python's garbage collector to run (clears RAM)
    gc.collect()

    # 3. Clear PyTorch's unused cached GPU memory (clears GPU RAM)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Memory cleared.")
    # --- 🧹 END: Memory Clearing Section ---
    # --------------------------------------------------


Found 11727 frames in '../dataset/2025/sampleFrame_4_out_16fps/lab_1':
../dataset/2025/sampleFrame_4_out_16fps/lab_1/img_000001.jpg
../dataset/2025/sampleFrame_4_out_16fps/lab_1/img_000002.jpg
../dataset/2025/sampleFrame_4_out_16fps/lab_1/img_000003.jpg
../dataset/2025/sampleFrame_4_out_16fps/lab_1/img_000004.jpg
../dataset/2025/sampleFrame_4_out_16fps/lab_1/img_000005.jpg
...
../dataset/2025/sampleFrame_4_out_16fps/lab_1/img_011727.jpg
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>] 11727/11727, 20.4 task/s, elapsed: 574s, ETA:     0s
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing Human Pose Estimation for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>] 11727/11727, 14.6

### Run this if the files is 4500<=

In [11]:
for folder in final_files:
    print(f"\n--- Memulai proses untuk folder: {folder} ---") # Menambah indikator awal loop
    vidio_path = f'/{folder}'
    target_dir = str(extracted_folder_path) + str(vidio_path)
    video = str(video_folder_path) + str(vidio_path) + '.mp4'
    num_ver = 4

    path_extractPose = f'../dataset/2025/2025_{num_ver}/extracted_pose'
    path_annData = f'../dataset/2025/2025_{num_ver}/annotated_data'
    Path(path_extractPose).mkdir(parents=True, exist_ok=True)
    Path(path_annData).mkdir(parents=True, exist_ok=True)

    ann_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.csv'
    pkl_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.pkl'
    out_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '_gt.mp4'

    if not osp.exists(target_dir):
        print(f"Error: Direktori tidak ada: {target_dir}")
        continue # Lanjut ke folder berikutnya jika direktori tidak ada
    else:
        search_pattern = osp.join(target_dir, 'img_*.jpg')
        frame_paths = glob.glob(search_pattern)
        frame_paths.sort()

        if frame_paths:
            print(f"Ditemukan {len(frame_paths)} frame di '{target_dir}'.")
        else:
            print(f"Tidak ada frame 'img_*.jpg' ditemukan di '{target_dir}'.")
            continue # Lanjut ke folder berikutnya jika tidak ada frame

    num_frame = len(frame_paths)
    try:
        original_frames = cv2.imread(frame_paths[0])
        h, w, _ = original_frames.shape
    except Exception as e:
        print(f"Error membaca frame pertama: {e}")
        continue # Lanjut ke folder berikutnya jika error

    print("Memulai deteksi manusia...")
    human_detections, _ = detection_inference(
        det_config,
        det_checkpoint,
        frame_paths,
        det_score_thr,
        device=device)
    torch.cuda.empty_cache() # Membersihkan cache setelah deteksi
    print("Deteksi manusia selesai.")

    print("Memulai estimasi pose...")
    pose_datasample = None
    pose_results, pose_datasample = pose_inference(
        pose_config,
        pose_checkpoint,
        frame_paths,
        human_detections,
        device=device)
    torch.cuda.empty_cache() # Membersihkan cache setelah pose
    print("Estimasi pose selesai.")

    stdet_preds = None

    print('Memulai deteksi aksi SpatioTemporal berbasis skeleton...')
    clip_len, frame_interval = predict_stepsize, 1

    timestamps, stdet_preds, skeleton_datasets = skeleton_based_stdet(
        predict_stepsize, video,
        human_detections,
        pose_results, num_frame,
        clip_len,
        frame_interval, h, w)
    
    # Normalisasi (jika diperlukan, sesuaikan dengan kode asli Anda)
    for i in range(len(human_detections)):
        det = human_detections[i]
        det[:, 0:4:2] *= 1
        det[:, 1:4:2] *= 1
        human_detections[i] = torch.from_numpy(det[:, :4]).to(device)

    print("Menyimpan anotasi dan data skeleton...")
    anno = ""
    if stdet_preds:
        for clip in stdet_preds:
            if clip is None:
                continue
            for person_attr in clip:
                anno += str(person_attr[0][0]) + "," + str(person_attr[0][1]) + "\n"

    with open(ann_filename, 'w') as data:
        data.write(anno)

    mmengine.dump(skeleton_datasets, pkl_filename)
    print("Penyimpanan selesai.")

    print("Membuat video hasil...")
    stdet_results = []
    if stdet_preds:
        for timestamp, prediction in zip(timestamps, stdet_preds):
            if timestamp - 1 < len(human_detections):
                human_detection = human_detections[timestamp - 1]
                stdet_results.append(
                    pack_result(human_detection, prediction, h, w))

    def dense_timestamps(timestamps, n):
        old_frame_interval = (timestamps[1] - timestamps[0]) if len(timestamps) > 1 else 0
        start = timestamps[0] - old_frame_interval / n * (n - 1) / 2
        new_frame_inds = np.arange(
            len(timestamps) * n) * old_frame_interval / n + start
        return new_frame_inds.astype(np.int64)

    dense_n = int(predict_stepsize / output_stepsize)
    # if timestamps: # Pastikan timestamps tidak kosong
    output_timestamps = dense_timestamps(timestamps, dense_n) + 1
    frames = [
        cv2.imread(frame_paths[timestamp - 1])
        for timestamp in output_timestamps if timestamp - 1 < len(frame_paths)
    ]
    
    pose_datasample_out = [
        pose_datasample[timestamp - 1] for timestamp in output_timestamps if timestamp - 1 < len(pose_datasample)
    ]

    vis_frames = visualize(pose_config, frames, stdet_results, pose_datasample_out, None)
    vid = mpy.ImageSequenceClip(vis_frames, fps=output_fps)
    vid.write_videofile(out_filename) # Tambah logger=None untuk output lebih bersih
    print(f"Video hasil disimpan di: {out_filename}")


    # --- PENAMBAHAN: Membersihkan Memori RAM dan GPU ---
    print(f"--- Membersihkan memori setelah folder: {folder} ---")
    
    # Hapus variabel yang tidak diperlukan lagi secara eksplisit (opsional tapi bisa membantu)
    del human_detections, pose_results, pose_datasample, stdet_preds, skeleton_datasets
    del stdet_results, frames, vis_frames, vid, original_frames, frame_paths
    
    # Membersihkan RAM (Garbage Collection)
    gc.collect() 
    print("Garbage collection dijalankan.")
    
    # Membersihkan Cache GPU (jika menggunakan PyTorch dan CUDA)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("Cache CUDA dibersihkan.")
    
    print(f"--- Selesai iterasi untuk folder: {folder} ---\n")


--- Memulai proses untuk folder: lab_1_1 ---
Ditemukan 4500 frame di '../dataset/2025/sampleFrame_4_out_16fps/lab_1_1'.
Memulai deteksi manusia...
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 4500/4500, 19.8 task/s, elapsed: 228s, ETA:     0s
Deteksi manusia selesai.
Memulai estimasi pose...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing Human Pose Estimation for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 4500/4500, 15.0 task/s, elapsed: 301s, ETA:     0s
Estimasi pose selesai.
Memulai deteksi aksi SpatioTemporal berbasis skeleton...
Building skeleton datasets from existing keypoint data for each clip
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 1125/1125, 1974.7 task/s, ela

Moviepy - Done !
Moviepy - video ready ../dataset/2025/2025_4/extracted_pose/lab_1_1_gt.mp4
Video hasil disimpan di: ../dataset/2025/2025_4/extracted_pose/lab_1_1_gt.mp4
--- Membersihkan memori setelah folder: lab_1_1 ---
Garbage collection dijalankan.
Cache CUDA dibersihkan.
--- Selesai iterasi untuk folder: lab_1_1 ---


--- Memulai proses untuk folder: lab_1_2 ---
Ditemukan 4500 frame di '../dataset/2025/sampleFrame_4_out_16fps/lab_1_2'.
Memulai deteksi manusia...
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 4500/4500, 22.5 task/s, elapsed: 200s, ETA:     0s
Deteksi manusia selesai.
Memulai estimasi pose...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing 

Moviepy - Done !
Moviepy - video ready ../dataset/2025/2025_4/extracted_pose/lab_1_2_gt.mp4
Video hasil disimpan di: ../dataset/2025/2025_4/extracted_pose/lab_1_2_gt.mp4
--- Membersihkan memori setelah folder: lab_1_2 ---
Garbage collection dijalankan.
Cache CUDA dibersihkan.
--- Selesai iterasi untuk folder: lab_1_2 ---


--- Memulai proses untuk folder: lab_1_3 ---
Ditemukan 2727 frame di '../dataset/2025/sampleFrame_4_out_16fps/lab_1_3'.
Memulai deteksi manusia...
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 2727/2727, 22.0 task/s, elapsed: 124s, ETA:     0s
Deteksi manusia selesai.
Memulai estimasi pose...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing 

Moviepy - Done !
Moviepy - video ready ../dataset/2025/2025_4/extracted_pose/lab_1_3_gt.mp4
Video hasil disimpan di: ../dataset/2025/2025_4/extracted_pose/lab_1_3_gt.mp4
--- Membersihkan memori setelah folder: lab_1_3 ---
Garbage collection dijalankan.
Cache CUDA dibersihkan.
--- Selesai iterasi untuk folder: lab_1_3 ---


--- Memulai proses untuk folder: lab_2_1 ---
Ditemukan 4500 frame di '../dataset/2025/sampleFrame_4_out_16fps/lab_2_1'.
Memulai deteksi manusia...
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 4500/4500, 22.6 task/s, elapsed: 199s, ETA:     0s
Deteksi manusia selesai.
Memulai estimasi pose...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing 

Moviepy - Done !
Moviepy - video ready ../dataset/2025/2025_4/extracted_pose/lab_2_1_gt.mp4
Video hasil disimpan di: ../dataset/2025/2025_4/extracted_pose/lab_2_1_gt.mp4
--- Membersihkan memori setelah folder: lab_2_1 ---
Garbage collection dijalankan.
Cache CUDA dibersihkan.
--- Selesai iterasi untuk folder: lab_2_1 ---


--- Memulai proses untuk folder: lab_2_2 ---
Ditemukan 4500 frame di '../dataset/2025/sampleFrame_4_out_16fps/lab_2_2'.
Memulai deteksi manusia...
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 4500/4500, 22.5 task/s, elapsed: 200s, ETA:     0s
Deteksi manusia selesai.
Memulai estimasi pose...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing 

...

Moviepy - Done !
Moviepy - video ready ../dataset/2025/2025_4/extracted_pose/lab_7_1_gt.mp4
Video hasil disimpan di: ../dataset/2025/2025_4/extracted_pose/lab_7_1_gt.mp4
--- Membersihkan memori setelah folder: lab_7_1 ---
Garbage collection dijalankan.
Cache CUDA dibersihkan.
--- Selesai iterasi untuk folder: lab_7_1 ---


--- Memulai proses untuk folder: lab_7_2 ---
Ditemukan 4500 frame di '../dataset/2025/sampleFrame_4_out_16fps/lab_7_2'.
Memulai deteksi manusia...
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 4500/4500, 22.5 task/s, elapsed: 200s, ETA:     0s
Deteksi manusia selesai.
Memulai estimasi pose...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing 

Moviepy - Done !
Moviepy - video ready ../dataset/2025/2025_4/extracted_pose/lab_7_2_gt.mp4
Video hasil disimpan di: ../dataset/2025/2025_4/extracted_pose/lab_7_2_gt.mp4
--- Membersihkan memori setelah folder: lab_7_2 ---
Garbage collection dijalankan.
Cache CUDA dibersihkan.
--- Selesai iterasi untuk folder: lab_7_2 ---


--- Memulai proses untuk folder: lab_7_3 ---
Ditemukan 4500 frame di '../dataset/2025/sampleFrame_4_out_16fps/lab_7_3'.
Memulai deteksi manusia...
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 4500/4500, 22.6 task/s, elapsed: 199s, ETA:     0s
Deteksi manusia selesai.
Memulai estimasi pose...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing 

Moviepy - Done !
Moviepy - video ready ../dataset/2025/2025_4/extracted_pose/lab_7_3_gt.mp4
Video hasil disimpan di: ../dataset/2025/2025_4/extracted_pose/lab_7_3_gt.mp4
--- Membersihkan memori setelah folder: lab_7_3 ---
Garbage collection dijalankan.
Cache CUDA dibersihkan.
--- Selesai iterasi untuk folder: lab_7_3 ---


--- Memulai proses untuk folder: lab_7_4 ---
Ditemukan 4500 frame di '../dataset/2025/sampleFrame_4_out_16fps/lab_7_4'.
Memulai deteksi manusia...
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 4500/4500, 22.0 task/s, elapsed: 204s, ETA:     0s
Deteksi manusia selesai.
Memulai estimasi pose...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing 

Moviepy - Done !
Moviepy - video ready ../dataset/2025/2025_4/extracted_pose/lab_7_4_gt.mp4
Video hasil disimpan di: ../dataset/2025/2025_4/extracted_pose/lab_7_4_gt.mp4
--- Membersihkan memori setelah folder: lab_7_4 ---
Garbage collection dijalankan.
Cache CUDA dibersihkan.
--- Selesai iterasi untuk folder: lab_7_4 ---


--- Memulai proses untuk folder: lab_7_5 ---
Ditemukan 3770 frame di '../dataset/2025/sampleFrame_4_out_16fps/lab_7_5'.
Memulai deteksi manusia...
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 3770/3770, 22.3 task/s, elapsed: 169s, ETA:     0s
Deteksi manusia selesai.
Memulai estimasi pose...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing 

Moviepy - Done !
Moviepy - video ready ../dataset/2025/2025_4/extracted_pose/lab_7_5_gt.mp4
Video hasil disimpan di: ../dataset/2025/2025_4/extracted_pose/lab_7_5_gt.mp4
--- Membersihkan memori setelah folder: lab_7_5 ---
Garbage collection dijalankan.
Cache CUDA dibersihkan.
--- Selesai iterasi untuk folder: lab_7_5 ---



In [14]:
print(f'{extracted_folder_path}{vidio_path}.pkl')

../dataset/2025/sampleFrame_4_out_16fps/lab_7.pkl


In [16]:
datas = mmengine.load(f'{extracted_folder_path}{vidio_path}.pkl')
sdet = datas['stdet_results']
pdet = datas['pose_datasample']

In [17]:
mmengine.dump(
    {
        'stdet_results': sdet,
        'pose_datasample': pdet,
        'output_timestamps': output_timestamps
    },
    f'{extracted_folder_path}{vidio_path}.pkl'
)

In [ ]:
data  = mmengine.load(f'{extracted_folder_path}{vidio_path}.pkl')

# Access each item
stdet_results = data['stdet_results']
pose_datasample = data['pose_datasample']
output_timestamps = data['output_timestamps']



[    1     2     3 ... 21768 21769 21770]


In [32]:
import os
import shutil
import math

def split_files_into_folders(source_folder_path, max_files_per_folder=4500):
    """
    Memisahkan file dalam folder sumber ke beberapa subfolder secara sekuensial
    berdasarkan nama file, dengan jumlah file maksimal tertentu.

    Args:
        source_folder_path (str): Path ke folder sumber.
        max_files_per_folder (int): Jumlah maksimal file per subfolder.
    """
    try:
        if not os.path.isdir(source_folder_path):
            print(f"Error: Folder sumber '{source_folder_path}' tidak ditemukan.")
            return

        all_items = os.listdir(source_folder_path)
        files_to_move = [
            f for f in all_items if os.path.isfile(os.path.join(source_folder_path, f))
        ]

        if not files_to_move:
            print(f"Tidak ada file yang ditemukan di '{source_folder_path}'.")
            return

        # --- TAMBAHAN: Mengurutkan daftar file secara alphabetical ---
        files_to_move.sort()
        print("Daftar file telah diurutkan berdasarkan nama.")
        # -----------------------------------------------------------

        num_files = len(files_to_move)
        print(f"Total file yang ditemukan: {num_files}")

        num_subfolders = math.ceil(num_files / max_files_per_folder)
        print(f"Akan dibuat {num_subfolders} subfolder.")

        base_folder_name = os.path.basename(os.path.normpath(source_folder_path))
        parent_folder = os.path.dirname(source_folder_path) # Dapatkan folder induk

        for i in range(num_subfolders):
            subfolder_name = f"{base_folder_name}_{i + 1}"
            # Pastikan subfolder dibuat di lokasi yang sama dengan folder sumber
            subfolder_path = os.path.join(parent_folder, subfolder_name) 

            os.makedirs(subfolder_path, exist_ok=True)
            print(f"Subfolder '{subfolder_path}' telah dibuat/ditemukan.")

            start_index = i * max_files_per_folder
            end_index = start_index + max_files_per_folder

            files_for_current_subfolder = files_to_move[start_index:end_index]

            for file_name in files_for_current_subfolder:
                source_file_path = os.path.join(source_folder_path, file_name)
                destination_file_path = os.path.join(subfolder_path, file_name)

                try:
                    shutil.move(source_file_path, destination_file_path)
                except Exception as e:
                    print(f"  Error saat memindahkan '{file_name}': {e}")
            
            print(f"Berhasil memindahkan {len(files_for_current_subfolder)} file ke '{subfolder_name}'.")

        print("\nProses pemisahan file selesai.")

    except Exception as e:
        print(f"Terjadi kesalahan: {e}")

if __name__ == "__main__":
    extracted_folder_path = "../dataset/2025/sampleFrame_4_out_16fps"
    video_folder_path = "../assets/video/25_5_v1"

    excluded_file = []
    final_files = sorted([
            f for f in os.listdir(extracted_folder_path)
            # Check if it IS a directory
            if os.path.isdir(os.path.join(extracted_folder_path, f)) 
            # AND check if its base name (without extension) is NOT excluded
            and os.path.splitext(f)[0] not in excluded_file
        ])
    for file in final_files:
        ff_path = f"{extracted_folder_path}/{file}"
        split_files_into_folders(ff_path)
    folder_path_input = input("Masukkan path ke folder yang ingin dipisah filenya: ")

Daftar file telah diurutkan berdasarkan nama.
Total file yang ditemukan: 11727
Akan dibuat 3 subfolder.
Subfolder '../dataset/2025/sampleFrame_4_out_16fps/lab_1_1' telah dibuat/ditemukan.
Berhasil memindahkan 4500 file ke 'lab_1_1'.
Subfolder '../dataset/2025/sampleFrame_4_out_16fps/lab_1_2' telah dibuat/ditemukan.
Berhasil memindahkan 4500 file ke 'lab_1_2'.
Subfolder '../dataset/2025/sampleFrame_4_out_16fps/lab_1_3' telah dibuat/ditemukan.
Berhasil memindahkan 2727 file ke 'lab_1_3'.

Proses pemisahan file selesai.
Daftar file telah diurutkan berdasarkan nama.
Total file yang ditemukan: 10237
Akan dibuat 3 subfolder.
Subfolder '../dataset/2025/sampleFrame_4_out_16fps/lab_2_1' telah dibuat/ditemukan.
Berhasil memindahkan 4500 file ke 'lab_2_1'.
Subfolder '../dataset/2025/sampleFrame_4_out_16fps/lab_2_2' telah dibuat/ditemukan.
Berhasil memindahkan 4500 file ke 'lab_2_2'.
Subfolder '../dataset/2025/sampleFrame_4_out_16fps/lab_2_3' telah dibuat/ditemukan.
Berhasil memindahkan 1237 file 

In [33]:
extracted_folder_path = "../dataset/2025/sampleFrame_4_out_16fps"
video_folder_path = "../assets/video/25_5_v1"

excluded_file = ['lab_7']
final_files = sorted([
        f for f in os.listdir(extracted_folder_path)
        # Check if it IS a directory
        if os.path.isdir(os.path.join(extracted_folder_path, f)) 
        # AND check if its base name (without extension) is NOT excluded
        and os.path.splitext(f)[0] not in excluded_file
    ])
for file in final_files:
    print(f"{extracted_folder_path}/{file}")

../dataset/2025/sampleFrame_4_out_16fps/lab_1_1
../dataset/2025/sampleFrame_4_out_16fps/lab_1_2
../dataset/2025/sampleFrame_4_out_16fps/lab_1_3
../dataset/2025/sampleFrame_4_out_16fps/lab_2_1
../dataset/2025/sampleFrame_4_out_16fps/lab_2_2
../dataset/2025/sampleFrame_4_out_16fps/lab_2_3
../dataset/2025/sampleFrame_4_out_16fps/lab_3_1
../dataset/2025/sampleFrame_4_out_16fps/lab_3_2
../dataset/2025/sampleFrame_4_out_16fps/lab_3_3
../dataset/2025/sampleFrame_4_out_16fps/lab_4_1
../dataset/2025/sampleFrame_4_out_16fps/lab_4_2
../dataset/2025/sampleFrame_4_out_16fps/lab_4_3
../dataset/2025/sampleFrame_4_out_16fps/lab_5_1
../dataset/2025/sampleFrame_4_out_16fps/lab_5_2
../dataset/2025/sampleFrame_4_out_16fps/lab_6_1
../dataset/2025/sampleFrame_4_out_16fps/lab_6_2
../dataset/2025/sampleFrame_4_out_16fps/lab_6_3
../dataset/2025/sampleFrame_4_out_16fps/lab_7_1
../dataset/2025/sampleFrame_4_out_16fps/lab_7_2
../dataset/2025/sampleFrame_4_out_16fps/lab_7_3
../dataset/2025/sampleFrame_4_out_16fps/